# 4. Variables que usan el target

La variable más fuerte es la tasa de usados de cada vendedor. Si al calcularla se incluye la etiqueta de la propia fila, el modelo la aprende y en test falla.

Por eso, en la sección de target encoding de `modelo.py`, cada fila de entrenamiento se codifica con etiquetas de otras filas, por sub-folds. Validación y test se codifican con las etiquetas de todo el entrenamiento.

In [1]:
import sys
import warnings

sys.path.insert(0, "..")  # Los módulos están en la raíz del proyecto
warnings.filterwarnings("ignore")
%matplotlib inline

import numpy as np
import pandas as pd

import datos
import modelo
import pruebas

crudo_train, crudo_test = datos.cargar_train_test()
train, test = datos.construir(crudo_train, crudo_test)

El suavizado acerca a la tasa global los grupos con pocas filas, porque su tasa merece menos confianza (D41). El vendedor B tiene un solo item y por eso queda más cerca de la media que el A.

In [2]:
ejemplo = pd.DataFrame({"seller_id": ["A", "A", "A", "A", "B", "C", "C"]})
etiquetas = pd.Series([1, 1, 1, 1, 1, 0, 0])
modelo.EncoderDeGrupo("seller_id").fit(ejemplo, etiquetas).transform(pd.DataFrame({"seller_id": ["A", "B", "C", "D"]}))

,te_seller_id,te_seller_id_n
0,0.904762,4.0
1,0.809524,1.0
2,0.357143,2.0
3,NaN,0.0


## Los encoders del modelo

Tasas por vendedor y por categoría, tasas por vendedor cruzado con otras variables, y dos variantes de vecinos, los títulos más parecidos de todo el catálogo y los items más parecidos del mismo vendedor.

In [3]:
[e.nombre if hasattr(e, "nombre") else type(e).__name__ for e in modelo.crear_encoders()]

['te_seller_id',
 'te_category_id',
 'EncoderDeVecinos',
 'te_seller_id_x_listing_type_id',
 'te_seller_id_x_iq_es_1',
 'te_seller_id_x_tiene_warranty',
 'te_seller_id_x_title',
 'EncoderVecinosDelVendedor']

## Qué producen

In [4]:
muestra = train.sample(pruebas.MUESTRA_TARGET_ENCODING, random_state=datos.SEED).reset_index(drop=True)
y = modelo.y_binario(muestra)
codificado, _ = modelo.codificar(muestra, y, muestra.head(1))
tabla = pd.DataFrame({
    "% nulos": 100 * codificado.isna().mean(),
    "correlación con el target": codificado.corrwith(y),
}).round(3)
tabla.sort_values("correlación con el target", ascending=False)

,% nulos,correlación con el target
te_vv_tasa_mismo_tipo_y_stock,54.447,0.915
te_seller_id_x_title,99.533,0.908
te_vv_tasa_precio_parecido,71.980,0.891
te_seller_id_x_listing_type_id,52.107,0.873
te_vv_tasa_ponderada,48.740,0.872
te_seller_id_x_iq_es_1,51.393,0.848
te_seller_id_x_tiene_warranty,51.580,0.844
te_vv_y_mas_parecido,48.740,0.841
te_seller_id,48.740,0.841
te_vv_y_mas_cercano_en_tiempo,48.740,0.838


## Controles

El primero invierte las etiquetas de un sub-fold y comprueba que sus filas no cambien. El segundo mezcla las etiquetas al azar, y si el modelo completo igual acertara, algo estaría filtrándose. Tiene que caer al nivel de la clase mayoritaria.

Una lista vacía significa que no se encontró ningún problema.

In [5]:
for control in [pruebas.te_sin_etiqueta_propia, pruebas.target_mezclado]:
    print(control.__doc__, "Problemas encontrados", control(crudo_train, crudo_test, train, test))

Invertir las etiquetas de un sub-fold no cambia el target encoding de sus filas Problemas encontrados []


      accuracy con target mezclado 0.508, clase mayoritaria 0.532
Con etiquetas al azar, el modelo completo cae al nivel de la clase mayoritaria Problemas encontrados []
